# Skill Memory vs. OCL Survey strategies

This notebook compares the native **Skill Memory** strategy with the OCL Survey strategies using the same result format and post-processing utilities.

The notebook is designed to be reproducible in Google Colab: setup and experiment execution are performed from notebook cells. Existing results for the other OCL Survey strategies are read when available. **Pre-existing Skill Memory benchmark results are not used for the comparison.**


## 0. Generate fresh Skill Memory results

Run the experiment directly from this notebook. You do **not** need to open a terminal or manually run `experiments/main.py` from the repository root.

The command below uses the same OCL Survey experiment pipeline as every other strategy. The `device=auto` configuration selects CUDA when a GPU is available and otherwise falls back to CPU.

**Run this cell only when you want to generate fresh Skill Memory results. Training can take a while.**

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/kobros-tech/ocl_survey.git"
COLAB_ROOT = Path("/content/ocl_survey")

# When the notebook is opened in Colab, make sure the repository is available.
if "google.colab" in sys.modules and not (COLAB_ROOT / "experiments" / "main.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)

REPO_ROOT = COLAB_ROOT if (COLAB_ROOT / "experiments" / "main.py").exists() else Path.cwd().resolve()
if not (REPO_ROOT / "experiments" / "main.py").exists():
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "experiments" / "main.py").exists():
            REPO_ROOT = parent
            break

print(f"Repository: {REPO_ROOT}")
print("Run the next cell to generate fresh Skill Memory results.")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT)

subprocess.run(
    [
        sys.executable,
        "experiments/main.py",
        "strategy=skill_memory",
        "experiment=split_cifar100",
    ],
    cwd=REPO_ROOT,
    env=env,
    check=True,
)


After the run finishes, continue with the comparison cells below. The fresh Skill Memory results are expected under `results/skill_memory_split_cifar100_20_2000/`.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.toolkit.process_results import extract_results
from src.toolkit.post_metrics import (
    compute_average_forgetting,
    compute_AAA,
    compute_wcacc,
)

RESULTS_ROOT = REPO_ROOT / "results"
BENCHMARK, NUM_TASKS, MEMORY_SIZE = "split_cifar100", 20, 2000

STRATEGIES = {
    "Skill Memory": "skill_memory",
    "ER": "er",
    "ER-ACE": "er_ace",
    "DER++": "der",
    "MIR": "mir",
    "ER + LwF": "er_lwf",
    "RAR": "rar",
    "SCR": "scr",
    "AGEM": "agem",
    "MER": "mer",
    "iCaRL": "icarl",
    "GDumb": "gdumb",
}

TEST_STREAM = "Top1_Acc_Stream/eval_phase/test_stream/Task000"
VALID_STREAM = "Top1_Acc_Stream/eval_phase/valid_stream/Task000"
TEST_EXP = "Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp"
VALID_EXP = "Top1_Acc_Exp/eval_phase/valid_stream/Task000/Exp"

sns.set_theme(style="whitegrid", context="notebook")


## 1. Load OCL Survey results

Results are discovered using `<strategy>_<benchmark>_<tasks>_<memory>`. Missing methods are reported rather than treated as zero.

In [ ]:
def result_name(prefix):
    return f"{prefix}_{BENCHMARK}_{NUM_TASKS}_{MEMORY_SIZE}"

result_dirs = {label: result_name(prefix) for label, prefix in STRATEGIES.items()}
available = {label: RESULTS_ROOT / name for label, name in result_dirs.items()
             if (RESULTS_ROOT / name).is_dir()}

print("Available results:")
for label, path in available.items():
    print(f"  ✓ {label}: {path}")

print("\nMissing results:")
for label, name in result_dirs.items():
    if label not in available:
        print(f"  - {label}: {RESULTS_ROOT / name}")

frames = {}
for label, path in available.items():
    try:
        frames[label] = extract_results(str(path), verbose=False)
    except Exception as exc:
        warnings.warn(f"Could not load {label}: {exc}")

pd.DataFrame([
    {
        "method": label,
        "seeds": sorted(frame.get("training", pd.DataFrame()).get("seed", pd.Series(dtype=int)).dropna().unique().tolist()),
    }
    for label, frame in frames.items()
])


## 2. Comparison metrics

The summary uses the repository's existing final accuracy, average forgetting, AAA, and WC-Acc implementations.

In [ ]:
summary = []

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty:
        continue

    required = [f"{TEST_EXP}{i:03d}" for i in range(NUM_TASKS)]
    if not set(required).issubset(df.columns):
        warnings.warn(f"Skipping metrics for {label}: missing task accuracy columns")
        continue

    forgetting_df = compute_average_forgetting(df.copy(), NUM_TASKS)
    final_forgetting = forgetting_df.loc[forgetting_df["mb_index"] == forgetting_df["mb_index"].max(), "Average_Forgetting"].dropna()
    final_accuracy = df.loc[df["mb_index"] == df["mb_index"].max(), TEST_STREAM].dropna()

    row = {
        "method": label,
        "final_accuracy": final_accuracy.mean(),
        "final_accuracy_std": final_accuracy.std(),
        "forgetting": final_forgetting.mean(),
        "forgetting_std": final_forgetting.std(),
        "AAA": np.nan,
        "AAA_std": np.nan,
        "WCAcc": np.nan,
        "WCAcc_std": np.nan,
    }

    try:
        aaa_df = compute_AAA(df.copy(), VALID_STREAM)
        aaa = aaa_df.loc[aaa_df["mb_index"] == aaa_df["mb_index"].max(), "AAA"].dropna()
        row["AAA"], row["AAA_std"] = aaa.mean(), aaa.std()
    except Exception as exc:
        warnings.warn(f"AAA unavailable for {label}: {exc}")

    try:
        wc_df = compute_wcacc(df.copy(), NUM_TASKS)
        wc = wc_df.loc[wc_df["mb_index"] == wc_df["mb_index"].max(), "WCAcc"].dropna()
        row["WCAcc"], row["WCAcc_std"] = wc.mean(), wc.std()
    except Exception as exc:
        warnings.warn(f"WC-Acc unavailable for {label}: {exc}")

    summary.append(row)

summary_df = pd.DataFrame(summary).sort_values("final_accuracy", ascending=False)
display(summary_df.style.format({
    "final_accuracy": "{:.2%}", "final_accuracy_std": "{:.2%}",
    "forgetting": "{:.2%}", "forgetting_std": "{:.2%}",
    "AAA": "{:.2%}", "AAA_std": "{:.2%}",
    "WCAcc": "{:.2%}", "WCAcc_std": "{:.2%}",
}))


## 3. Online accuracy

The curve is smoothed per seed and then summarized across seeds.

In [ ]:
window = 10
fig, ax = plt.subplots(figsize=(12, 7))

for label, result in frames.items():
    df = result.get("training")
    if df is None or TEST_STREAM not in df.columns:
        continue
    x = df[["seed", "mb_index", TEST_STREAM]].dropna().sort_values(["seed", "mb_index"]).copy()
    x["smooth"] = x.groupby("seed")[TEST_STREAM].transform(lambda s: s.rolling(window, min_periods=1).mean())
    g = x.groupby("mb_index")["smooth"].agg(["mean", "std"]).reset_index()
    ax.plot(g["mb_index"], 100 * g["mean"], label=label)
    ax.fill_between(g["mb_index"], 100 * (g["mean"] - g["std"]), 100 * (g["mean"] + g["std"]), alpha=0.10)

ax.set(xlabel="Batch index", ylabel="Test-stream accuracy (%)", title=f"{BENCHMARK}: online accuracy")
ax.legend(ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. Catastrophic forgetting over training

This is the evolution of **average forgetting** across the 20 tasks as training progresses. It uses the same `compute_average_forgetting` utility used by the scalar comparison metric above. Lower is better: increasing values mean previously learned task accuracy is being lost.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty:
        continue

    required = [f"{TEST_EXP}{i:03d}" for i in range(NUM_TASKS)]
    if not set(required).issubset(df.columns):
        warnings.warn(f"Skipping forgetting curve for {label}: missing task accuracy columns")
        continue

    forgetting_df = compute_average_forgetting(df.copy(), NUM_TASKS)
    x = forgetting_df[["seed", "mb_index", "Average_Forgetting"]].dropna()
    if x.empty:
        continue

    g = x.groupby("mb_index")["Average_Forgetting"].agg(["mean", "std"]).reset_index()
    ax.plot(g["mb_index"], 100 * g["mean"], label=label)
    ax.fill_between(g["mb_index"], 100 * (g["mean"] - g["std"]), 100 * (g["mean"] + g["std"]), alpha=0.10)

ax.axhline(0, linewidth=1)
ax.set(xlabel="Batch index", ylabel="Average forgetting (%)", title=f"{BENCHMARK}: catastrophic forgetting over training")
ax.legend(ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 5. Reproducibility

Skill Memory results are generated by the notebook through the same OCL Survey experiment entry point used by the other strategies. The comparison does not use old Skill Memory results. For a fair comparison, keep the benchmark, task count, memory size, model, optimizer, evaluation settings, and seeds consistent across methods.

In [ ]:
analysis_dir = RESULTS_ROOT / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
summary_path = analysis_dir / "skill_memory_strategy_comparison.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")
